# Phase 9 — LLM explanation: grounded vs ungrounded

The same local model (`llama3.1:8b`) explains two findings, first WITHOUT graph facts (ungrounded) and then WITH the retrieved subgraph (grounded). Requires Ollama running and the model pulled.

- **openssh 9.6p1 / CVE-2024-6387** — genuinely affected (RCE).
- **zlib 1.3.1 / CVE-2022-37434** — NOT affected (fixed in 1.2.13) — the false-positive test.

In [9]:
import sys; sys.path.append('../src')
import ssc_graph as kg, graphrag, llm

g = kg.load(['../data/uthp_build.ttl', '../data/uthp_context.ttl', '../data/uthp_intel.ttl'])
openssh = graphrag.component_iri(g, 'openssh')
zlib    = graphrag.component_iri(g, 'zlib')
print('Retrieved context (grounding) for each case:\n')
print('--- openssh ---'); print(graphrag.context_text(g, openssh))
print('\n--- zlib ---'); print(graphrag.context_text(g, zlib))

Retrieved context (grounding) for each case:

--- openssh ---
Component: pkg_openssh_9_6p1 version 9.6p1.
Vulnerability match: CVE-2024-6387 — weakness Signal Handler Race Condition — severity High (CVSS 8.1) — VEX status: Affected — risk RiskHigh.
Impact path: pkg_openssh_9_6p1 runs on uthp_gateway, which connects to j1939_can, which reaches brake_ecu, which supports braking (a safety function).
Source of intelligence: local NVD snapshot.

--- zlib ---
Component: pkg_zlib_1_3_1 version 1.3.1.
Vulnerability match: CVE-2022-37434 — weakness Out-of-bounds Write — severity Critical (CVSS 9.8) — VEX status: NotAffected — risk None.
Impact path: pkg_zlib_1_3_1 runs on uthp_gateway, which connects to j1939_can, which reaches brake_ecu, which supports braking (a safety function).
Source of intelligence: local NVD snapshot.


### Case 1 — openssh (truly affected)

In [10]:
print('UNGROUNDED:\n', llm.explain_ungrounded('openssh', '9.6p1', 'CVE-2024-6387'))
print('\nGROUNDED:\n', llm.explain_grounded(g, openssh))

UNGROUNDED:
 Based on publicly available information:

1. The "openssh" version "9.6p1" is not directly related to CVE-2024-6387, as the vulnerability was reported in OpenSSH versions prior to 9.5.
2. A vehicle's safety-critical systems should not be connected to the internet or exposed to potential SSH vulnerabilities. However, if a vehicle's software package includes an outdated version of "openssh", it may pose a risk to the vehicle's network and potentially allow unauthorized access.

GROUNDED:
 Here are the answers based on the provided facts:

1. The component is affected by the vulnerability because its VEX status is Affected.

2. If affected, the potential impact on the vehicle could be a loss of braking capability due to the Signal Handler Race Condition in pkg_openssh_9_6p1 affecting brake_ecu's ability to support braking (a safety function).

3. Evidence source: local NVD snapshot.


### Case 2 — zlib (NOT affected — the false positive)

In [11]:
print('UNGROUNDED:\n', llm.explain_ungrounded('zlib', '1.3.1', 'CVE-2022-37434'))
print('\nGROUNDED:\n', llm.explain_grounded(g, zlib))

UNGROUNDED:
 Based on publicly available information:

1. Yes, zlib version 1.3.1 is affected by CVE-2022-37434.
2. The vulnerability allows an attacker to cause a denial-of-service (DoS) or potentially execute arbitrary code, which could lead to loss of vehicle control or other safety-critical system failures.

Note: I'm not aware of any specific information about the impact on vehicles, but in general, such vulnerabilities can have significant consequences for safety-critical systems.

GROUNDED:
 Here are the answers based on the provided facts:

1. The component is NOT affected by the vulnerability because its VEX status is NotAffected.

2. Since the component is not affected, there is no potential impact on the vehicle following the impact path.

3. Evidence source: Local NVD snapshot (National Vulnerability Database).


## What to look for
- **Grounded openssh**: should confirm AFFECTED and describe the CAN → brake-ECU → braking impact path, citing the snapshot — an explanation an ungrounded model cannot produce because it doesn't know the deployment.
- **Grounded zlib**: should say **NOT affected** (version 1.3.1 ≥ fixed 1.2.13).
- **Ungrounded zlib**: often calls it vulnerable (it doesn't know the fixed version) — the false positive your grounding suppresses.

This qualitative contrast is the worked example; **Phase 10** turns it into the quantitative ablation (precision/recall/false-positive rate over a labeled set).